In [6]:
import pandas as pd
import joblib
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.metrics import accuracy_score

# Load data
df = pd.read_csv(r"C:\Users\HP\OneDrive\Desktop\Fast-API\insurance (2).csv")
df_feat = df.copy()

# Feature engineering
df_feat["bmi"] = df_feat["weight"] / (df_feat["height"] ** 2)

def age_group(age):
    if age < 25:
        return "young"
    elif age < 45:
        return "adult"
    elif age < 60:
        return "middle_aged"
    return "senior"

df_feat["age_group"] = df_feat["age"].apply(age_group)

def lifestyle_risk(row):
    if row["smoker"] and row["bmi"] > 30:
        return "high"
    elif row["smoker"] or row["bmi"] > 27:
        return "medium"
    return "low"

df_feat["lifestyle_risk"] = df_feat.apply(lifestyle_risk, axis=1)

tier_1_cities = ["Mumbai", "Delhi", "Bangalore", "Chennai", "Kolkata", "Hyderabad", "Pune"]
tier_2_cities = ["Jaipur", "Chandigarh", "Indore", "Lucknow", "Patna", "Ranchi"]

def city_tier(city):
    if city in tier_1_cities:
        return 1
    elif city in tier_2_cities:
        return 2
    return 3

df_feat["city_tier"] = df_feat["city"].apply(city_tier)

# Select features & target
X = df_feat[[
    "bmi",
    "age_group",
    "lifestyle_risk",
    "city_tier",
    "income_lpa",
    "occupation"
]]
y = df_feat["insurance_premium_category"]

# Correct feature grouping
categorical_features = ["age_group", "lifestyle_risk", "occupation","city_tier"]
numeric_features = ["bmi", "income_lpa"]

preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
        ("num", "passthrough", numeric_features)
    ]
)

pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(random_state=42))
])

# Train
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=1
)
pipeline.fit(X_train, y_train)

# Evaluate
y_pred = pipeline.predict(X_test)
print("Accuracy:", accuracy_score(y_test, y_pred))




Accuracy: 0.85


In [ ]:
# Save model (CORRECT WAY)
joblib.dump(pipeline, "model.joblib")
print("✅ Model saved as model.joblib")

✅ Model saved as model.joblib


: 